# Movie Recommendation System
A content-based recommender system that builts using the TMDB 5000 Movie Dataset.
It recommends movies similar to the one you like based in genres, cast, director,and keywords.

## Step 1: Install Required Libraries

In [7]:
# Install all the required libraries
import subprocess
subprocess.run(["pip", "install", "pandas", "numpy", "scikit-learn", "ast"], 
               capture_output=True)

CompletedProcess(args=['pip', 'install', 'pandas', 'numpy', 'scikit-learn', 'ast'], returncode=1, stdout=b"Requirement already satisfied: pandas in c:\\users\\yadav\\appdata\\local\\programs\\python\\python313\\lib\\site-packages (3.0.1)\r\nRequirement already satisfied: numpy in c:\\users\\yadav\\appdata\\local\\programs\\python\\python313\\lib\\site-packages (2.4.3)\r\nRequirement already satisfied: scikit-learn in c:\\users\\yadav\\appdata\\local\\programs\\python\\python313\\lib\\site-packages (1.8.0)\r\nCollecting ast\r\n  Using cached AST-0.0.2.tar.gz (19 kB)\r\n  Installing build dependencies: started\r\n  Installing build dependencies: finished with status 'done'\r\n  Getting requirements to build wheel: started\r\n  Getting requirements to build wheel: finished with status 'error'\r\n", stderr=b'  error: subprocess-exited-with-error\r\n  \r\n  \xc3\x97 Getting requirements to build wheel did not run successfully.\r\n  \xe2\x94\x82 exit code: 1\r\n  \xe2\x95\xb0\xe2\x94\x80> [2

## Import Libraries


In [8]:
 #─── Core Libraries 
import pandas as pd          # For loading and manipulating data (like Excel for Python)
import numpy as np           # For numerical operations
import ast                   # For converting strings like "[{'id':1}]" into actual Python lists

# ─── Machine Learning 
from sklearn.feature_extraction.text import CountVectorizer
# CountVectorizer converts text into a matrix of word counts.
# Example: "action comedy" → [1, 1, 0, 0, ...]

from sklearn.metrics.pairwise import cosine_similarity
# Cosine similarity measures how similar two vectors are (0 = not similar, 1 = identical)

# ─── Save/Load Model 
import pickle  # Saves Python objects to disk so we don't recompute every time

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


#Step 3: Download and load the Dataset
We use the TMDB 5000 Movie Dataset which contains information about 5000 movies
including their genres, cast, crew, keywords, and plot descriptions.

### How to Download the Dataset
1. Go to: https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata
2. Click the Download button (you need a free Kaggle account)d

In [9]:
# LOAD BOTH CSV FILES
movies = pd.read_csv("tmdb_5000_movies.csv")
# This loads the main movie information:
# title, genres, keywords, overview, budget, revenue, etc.

credits = pd.read_csv("tmdb_5000_credits.csv")
# This loads the cast and crew information:
# movie_id, title, cast (actors), crew (director, producer etc.)

print("Movies dataset:")
print(f"  Rows: {movies.shape[0]}, Columns: {movies.shape[1]}")

print("\nCredits dataset:")
print(f"  Rows: {credits.shape[0]}, Columns: {credits.shape[1]}")


Movies dataset:
  Rows: 4803, Columns: 20

Credits dataset:
  Rows: 4803, Columns: 4


In [10]:
# ─── PREVIEW THE MOVIES DATASET 
movies.head(2)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [11]:
#  ─── PREVIEW THE CREDITS DATASET
credits.head(2)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [12]:
# # ─── MERGE BOTH DATASETS INTO ONE 
movies = movies.merge(credits, on="title")
#After merging 
# -Each row still represents one movie but now it has all the columns from both the files combined
print(f"After merging:")
print(f"  Rows: {movies.shape[0]}, Columns: {movies.shape[1]}")

# See all column names available after merge
print(f"\nAll columns:")
print(movies.columns.tolist())


After merging:
  Rows: 4809, Columns: 23

All columns:
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count', 'movie_id', 'cast', 'crew']


### Step 4: Select Relevent Columns
Out of 23 columns we only need 7 for our recommendation system.
We keep: movie_id, title, overview, genres, keywords, cast, crew

In [13]:
# Select only the columns we need

movies = movies[["movie_id",   # unique number identifying each movie
                 "title",      # movie name (shown in results)
                 "overview",   # short plot description in plain text
                 "genres",     # list of genres e.g. Action, Comedy, Drama
                 "keywords",   # list of theme keywords e.g. space, robot, love
                 "cast",       # list of actors in the movie
                 "crew"]]      # list of crew members (we only want the director)

print(f"Shape after selecting columns: {movies.shape}")

movies.head(2)


Shape after selecting columns: (4809, 7)


,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [14]:
# Check for the missing values 
print("Missing values in each column:")
print(movies.isnull().sum())

Missing values in each column:
movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64


In [15]:
# Remove the rows with the missing values
movies.dropna(inplace=True)

print(f"Shape after removing missing values: {movies.shape}")


Shape after removing missing values: (4806, 7)


## Step 5: Feature Extraction 
The genres, keywords, cast, and crew columns are stored as strings that look like lists. 
We need to convert them into actual Python lists so we can work with them.

In [16]:
# SEE THE RAW FORMAT OF GENRES COLUMN
print("Type of genres value:", type(movies["genres"].iloc[0]))
print("\nRaw genres value:")
print(movies["genres"].iloc[0])


Type of genres value: <class 'str'>

Raw genres value:
[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]


In [17]:
# CONVERT GENRES AND KEYWORDS
# this function takes the raw string and extracts just the names
def convert(text):
    """
    Input:  "[{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}]"
    Output: ['Action', 'Adventure']
    
    How it works:
    1. ast.literal_eval(text) converts the string into a real Python list of dicts
    2. We loop through each dictionary in the list
    3. We extract only the 'name' value from each dictionary
    4. We collect all names into a new list and return it
    """
    L = []                              # empty list to collect names
    for item in ast.literal_eval(text): # convert string to list, then loop
        L.append(item["name"])          # grab only the 'name' from each dict
    return L                            # return the clean list of names

#  APPLY THE FUNCTION TO GENRES AND KEYWORDS COLUMNS
movies["genres"]   = movies["genres"].apply(convert)
movies["keywords"] = movies["keywords"].apply(convert)

print("Genres after conversion:")
print(movies["genres"].iloc[0])

print("\nKeywords after conversion:")
print(movies["keywords"].iloc[0])


Genres after conversion:
['Action', 'Adventure', 'Fantasy', 'Science Fiction']

Keywords after conversion:
['culture clash', 'future', 'space war', 'space colony', 'society', 'space travel', 'futuristic', 'romance', 'space', 'alien', 'tribe', 'alien planet', 'cgi', 'marine', 'soldier', 'battle', 'love affair', 'anti war', 'power relations', 'mind and soul', '3d']


In [18]:
# FUNCTION TO EXTRACT TOP 3 CAST MEMBERS
def convert_cast(text):
    """
    Input:  "[{'name':'Sam Worthington',...}, {'name':'Zoe Saldana',...}, ...]"
    Output: ['SamWorthington', 'ZoeSaldana', 'SigourneyWeaver']
    (Only top 3 actors, and spaces removed from names — explained later)
    """
    L = []
    counter = 0
    for item in ast.literal_eval(text):  # loop through all cast members
        if counter < 3:                  # only take the first 3
            L.append(item["name"])
            counter += 1
        else:
            break                        # stop after 3, ignore the rest
    return L

movies["cast"] = movies["cast"].apply(convert_cast)

print("Cast after conversion (top 3 only):")
print(movies["cast"].iloc[0])


Cast after conversion (top 3 only):
['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']


In [19]:
# FUNCTION TO EXTRACT DIRECTOR FROM CREW
# The crew column has hundreds of people: directors, producers, writers, sound designers,etc.
# We only want the director.
def fetch_director(text):
    """
    Input:  "[{'job':'Director','name':'James Cameron'}, {'job':'Producer',...}]"
    Output: ['JamesCameron']
    
    We loop through crew members and return only the one with job == 'Director'
    """
    L = []
    for item in ast.literal_eval(text):   # loop through all crew members
        if item["job"] == "Director":      # check if this person is the director
            L.append(item["name"])
            break                          # only one director needed, stop looping
    return L

movies["crew"] = movies["crew"].apply(fetch_director)

print("Director after extraction:")
print(movies["crew"].iloc[0])


Director after extraction:
['James Cameron']


In [20]:
# CONVERT OVERVIEW FROM STRING TO LIST OF WORDS
movies["overview"] = movies["overview"].apply(lambda x: x.split())

print("Overview after splitting into words:")
print(movies["overview"].iloc[0][:8])  # show first 8 words only

Overview after splitting into words:
['In', 'the', '22nd', 'century,', 'a', 'paraplegic', 'Marine', 'is']


## Step 6: Remove Spaces from Multi-Word Names
Names like "Sam Mendes" must become "SamMendes" (one word).
Reason: If we keep the space, the model treats "Sam" and "Mendes" as two separate words.
This could accidentally match "Sam Raimi" with "Sam Mendes" just because both are named Sam.
By joining into one word, the full name becomes a single unique token.

In [21]:
# REMOVE SPACES FORM MULTI-WORD NAMES
movies["genres"]   = movies["genres"].apply(
                         lambda x: [i.replace(" ", "") for i in x])

movies["keywords"] = movies["keywords"].apply(
                         lambda x: [i.replace(" ", "") for i in x])

movies["cast"]     = movies["cast"].apply(
                         lambda x: [i.replace(" ", "") for i in x])

movies["crew"]     = movies["crew"].apply(
                         lambda x: [i.replace(" ", "") for i in x])

print("Genres after removing spaces:")
print(movies["genres"].iloc[0])

print("\nCast after removing spaces:")
print(movies["cast"].iloc[0])


Genres after removing spaces:
['Action', 'Adventure', 'Fantasy', 'ScienceFiction']

Cast after removing spaces:
['SamWorthington', 'ZoeSaldana', 'SigourneyWeaver']


### Step 7: Create the Tags Column
 We now combine overview + genres + keywords + cast +crew into one single text column called 'tags'.
 This single column will be what our model learns from.

In [22]:
# COMBINE ALL THE FEATURES INTO ONE 'TAGS' COLUMN

movies["tags"] = (movies["overview"]  +   # plot words
                  movies["genres"]    +   # genre names
                  movies["keywords"]  +   # keyword names
                  movies["cast"]      +   # top 3 actor names
                  movies["crew"])         # director name

print("Tags for first movie (as list):")
print(movies["tags"].iloc[0])


Tags for first movie (as list):
['In', 'the', '22nd', 'century,', 'a', 'paraplegic', 'Marine', 'is', 'dispatched', 'to', 'the', 'moon', 'Pandora', 'on', 'a', 'unique', 'mission,', 'but', 'becomes', 'torn', 'between', 'following', 'orders', 'and', 'protecting', 'an', 'alien', 'civilization.', 'Action', 'Adventure', 'Fantasy', 'ScienceFiction', 'cultureclash', 'future', 'spacewar', 'spacecolony', 'society', 'spacetravel', 'futuristic', 'romance', 'space', 'alien', 'tribe', 'alienplanet', 'cgi', 'marine', 'soldier', 'battle', 'loveaffair', 'antiwar', 'powerrelations', 'mindandsoul', '3d', 'SamWorthington', 'ZoeSaldana', 'SigourneyWeaver', 'JamesCameron']


In [23]:
# CONVERT TAGS LIST INTO SINGLE LOWERCASE STRING
movies["tags"] = movies["tags"].apply(lambda x: " ".join(x).lower())

print("Tags for first movie (as string):")
print(movies["tags"].iloc[0][:300])  # show first 300 characters


Tags for first movie (as string):
in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance spac


In [24]:
#CREATE FINAL CLEAN DATAFRAME
# We only need 3 columns for the actual recommendation model:
# modie_id - to uniquely identify movies
# title - to display movie names in results
# tags - the combined text our model learns from

new_df = movies[["movie_id", "title", "tags"]].copy()

print(f"Final dataframe shape: {new_df.shape}")
print("\nFirst 3 rows:")
new_df.head(3)


Final dataframe shape: (4806, 3)

First 3 rows:


,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...


## Step 8: Vectorize the Text Using CountVectorizer
We convert the 'tags' text column into numbers.
Each movie becomes a vector (a list of 5000 numbers).
Each number represents how many times a particular word appears in that movie's tags.

In [25]:
# CREATE THE COUNTVECTORIZER
cv = CountVectorizer(max_features=5000, stop_words="english")

vectors = cv.fit_transform(new_df["tags"]).toarray()

print(f"Vectors shape: {vectors.shape}")

print(f"\nVocabulary size: {len(cv.vocabulary_)}")

print(f"\nSample vector for first movie (first 20 values):")
print(vectors[0][:20])




Vectors shape: (4806, 5000)

Vocabulary size: 5000

Sample vector for first movie (first 20 values):
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


## Step 9: Compute Cosine Similarity
Now we calculate how similar every movie is to every other movie.
Result: a 4800 x 4800 matrix where each cell contains a similarity score.
Score closer to 1 = very similar. Score closer to 0 = very different

In [26]:
# COMPUTE COSINE SIMILARITY
# It takes our vectors matrix and computes teh similarity

similarity = cosine_similarity(vectors)

print(f"Similarity matrix shape: {similarity.shape}")

print(f"\nSimilarity scores of movie 0 vs first 5 movies:")
print(similarity[0][:5])



Similarity matrix shape: (4806, 4806)

Similarity scores of movie 0 vs first 5 movies:
[1.         0.08740748 0.05827165 0.03823596 0.17734311]


## Step 10: Build the Recommendation Function
This is the function you will call to get movie recommendations.
Give it a movie title and it returns the 5 most similar movies.

In [27]:
# THE RECOMMEND FUNCTION
def recommend(movie):
    """
    Given a movie title, returns the top 5 most similar movies.
    
    Parameters:
        movie (str): The exact title of a movie in our dataset
    
    Returns:
        Prints the top 5 recommended movie titles with similarity scores
    
    How it works step by step:
    1. Find the row number (index) of the input movie in our dataframe
    2. Get that movie's similarity scores against ALL 4800 other movies
    3. Sort all movies by their similarity score from highest to lowest
    4. Skip the first result (it's always the movie itself with score 1.0)
    5. Return the next 5 results — these are the most similar movies
    """
    # STEP 1: Find the index (row number) of the input movie
    movie_index = new_df[new_df["title"] == movie].index[0]
    
    print(f"Finding movies similar to: '{movie}' (index: {movie_index})")
    
    # STEP 2: Get similarity scores for this specific movie
    distances = similarity[movie_index]
    
    # STEP 3: Sort by similarity score (highest first)
    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )
    
    # STEP 4 & 5: Skip index 0 (the movie itself), take next 5
    # [1:6] means: start from position 1 (skip 0), take up to position 6
    top_5 = movies_list[1:6]
    
    # STEP 6: Print the results
    print(f"\n🎬 Because you liked '{movie}', you might enjoy:\n")
    for rank, (index, score) in enumerate(top_5, 1):
        title = new_df.iloc[index].title   # get the movie title using its index
        print(f"  {rank}. {title}")
        print(f"     Similarity score: {score:.4f}")
        print()
# TEST THE FUNCTION
recommend("Avatar")

Finding movies similar to: 'Avatar' (index: 0)

🎬 Because you liked 'Avatar', you might enjoy:

  1. Titan A.E.
     Similarity score: 0.2504

  2. Small Soldiers
     Similarity score: 0.2478

  3. Independence Day
     Similarity score: 0.2428

  4. Ender's Game
     Similarity score: 0.2410

  5. Aliens vs Predator: Requiem
     Similarity score: 0.2394



In [28]:
recommend("The Dark Knight")

Finding movies similar to: 'The Dark Knight' (index: 65)

🎬 Because you liked 'The Dark Knight', you might enjoy:

  1. The Dark Knight Rises
     Similarity score: 0.4239

  2. Batman Begins
     Similarity score: 0.3980

  3. Batman Returns
     Similarity score: 0.3216

  4. Batman Forever
     Similarity score: 0.2879

  5. Batman & Robin
     Similarity score: 0.2679



In [29]:
recommend("Interstellar")

Finding movies similar to: 'Interstellar' (index: 95)

🎬 Because you liked 'Interstellar', you might enjoy:

  1. Silent Running
     Similarity score: 0.2331

  2. Space Pirate Captain Harlock
     Similarity score: 0.2284

  3. The Martian
     Similarity score: 0.2223

  4. Apollo 13
     Similarity score: 0.2198

  5. Moonraker
     Similarity score: 0.2190



In [30]:
recommend("The Avengers")

Finding movies similar to: 'The Avengers' (index: 16)

🎬 Because you liked 'The Avengers', you might enjoy:

  1. Avengers: Age of Ultron
     Similarity score: 0.3627

  2. Captain America: Civil War
     Similarity score: 0.3290

  3. Iron Man 3
     Similarity score: 0.3122

  4. Captain America: The First Avenger
     Similarity score: 0.2982

  5. Iron Man
     Similarity score: 0.2920



In [31]:
recommend("The Godfather")

Finding movies similar to: 'The Godfather' (index: 3340)

🎬 Because you liked 'The Godfather', you might enjoy:

  1. Desert Dancer
     Similarity score: 0.5101

  2. Take the Lead
     Similarity score: 0.3947

  3. Step Up 2: The Streets
     Similarity score: 0.3408

  4. Center Stage
     Similarity score: 0.3366

  5. Footloose
     Similarity score: 0.3300



## ## Step 11: Save the Model Using Pickle
We save tw files to disk:
-movie.pkl :- the dataframe with movie_id,title,tags
-similaruty.phl :- the 4800x4800 similarity matrix
Next time we want recommendation, we just load these files instead of returning all the above steps(which takes less time) 

In [32]:
# SAVE MOVIES DATAFRAME

pickle.dump(new_df, open("movies.pkl", "wb"))

print("✅ movies.pkl saved successfully")
print(f"   Contains: {new_df.shape[0]} movies with columns: {new_df.columns.tolist()}")

✅ movies.pkl saved successfully
   Contains: 4806 movies with columns: ['movie_id', 'title', 'tags']


In [33]:
# SAVE SIMILARITY MATRIX

pickle.dump(similarity, open("similarity.pkl", "wb"))

print("✅ similarity.pkl saved successfully")
print(f"   Matrix size: {similarity.shape[0]} x {similarity.shape[1]}")
print(f"   Each cell contains a similarity score between two movies")

✅ similarity.pkl saved successfully
   Matrix size: 4806 x 4806
   Each cell contains a similarity score between two movies


In [34]:
# VERIFY THE SAVED FILES CORRECTLY

movies_loaded     = pickle.load(open("movies.pkl",     "rb"))
similarity_loaded = pickle.load(open("similarity.pkl", "rb"))

print("✅ Both files loaded successfully!")
print(f"\nmovies_loaded shape:     {movies_loaded.shape}")
print(f"similarity_loaded shape: {similarity_loaded.shape}")

# Quick test to confirm the loaded model works
print("\n--- Quick test with loaded model ---")
test_index = movies_loaded[movies_loaded["title"] == "Avatar"].index[0]
scores     = similarity_loaded[test_index]
top        = sorted(list(enumerate(scores)), reverse=True, key=lambda x: x[1])[1:4]
print("Top 3 movies similar to Avatar (using loaded model):")
for idx, score in top:
    print(f"  {movies_loaded.iloc[idx].title} ({score:.4f})")
    

✅ Both files loaded successfully!

movies_loaded shape:     (4806, 3)
similarity_loaded shape: (4806, 4806)

--- Quick test with loaded model ---
Top 3 movies similar to Avatar (using loaded model):
  Titan A.E. (0.2504)
  Small Soldiers (0.2478)
  Independence Day (0.2428)


## Step 12 (Bonus): Create a Streamlit Web App
We create a simple web app so anyone can use the recommender through a browser.

In [37]:
# app
app_code = '''import streamlit as st
import pickle
import pandas as pd

st.set_page_config(
    page_title="Movie Recommender",
    page_icon="Movie Recommender",
    layout="centered"
)

@st.cache_data
def load_data():
    movies_list = pickle.load(open("movies.pkl", "rb"))
    similarity  = pickle.load(open("similarity.pkl", "rb"))
    return movies_list, similarity

movies_list, similarity = load_data()

def recommend(movie):
    movie_index   = movies_list[movies_list["title"] == movie].index[0]
    distances     = similarity[movie_index]
    movies_sorted = sorted(list(enumerate(distances)),
                           reverse=True,
                           key=lambda x: x[1])[1:6]
    return [movies_list.iloc[i[0]].title for i in movies_sorted]

st.title("Movie Recommendation System")
st.markdown("---")
st.markdown("Select a movie you like and get 5 similar movie recommendations instantly.")

selected_movie_name = st.selectbox(
    "Choose a movie:",
    movies_list["title"].values
)

if st.button("Get Recommendations", use_container_width=True):
    recommendations = recommend(selected_movie_name)
    st.markdown("---")
    st.subheader(f"Because you liked {selected_movie_name}, try these:")
    st.markdown("")
    for i, movie_name in enumerate(recommendations, 1):
        st.markdown(f"**{i}.** {movie_name}")

st.markdown("---")
st.caption("Built with Python, scikit-learn, and Streamlit | TMDB 5000 Dataset")
'''

# ── THE FIX IS HERE ───────────────────────────────────────────────────────────
# encoding="utf-8" tells Python to save the file using UTF-8
# instead of Windows default cp1252 encoding
# This allows all characters to be saved correctly

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py created successfully in your project folder!")
print()
# 1. Open the VS Code terminal
# 2. Install streamlit by running:  pip install streamlit
# 3. Then run:  streamlit run app.py
# 4. Browser opens automatically

app.py created successfully in your project folder!

